<a href="https://colab.research.google.com/github/pedrosampaiom2007-gif/GS-AI-Space-Monitoring/blob/main/GS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Mission Control AI — Brazil Space Monitoring

**FIAP — Global Solution 2026.1 | Prompt and Artificial Intelligence**

Sistema inteligente de monitoramento e controle de missão espacial experimental.  
Monitora temperatura, comunicação, energia, oxigênio e estabilidade com geração automática de alertas e análise por IA (Llama 3.2 via Ollama).

---
**Execute as células em ordem. O modelo Llama será instalado automaticamente.**

In [1]:
# ============================================================
# CÉLULA 1 — INSTALAÇÃO E CONFIGURAÇÃO DO AMBIENTE
# ============================================================
import subprocess, time

print(' Instalando dependências...')
subprocess.run(['apt-get', 'update', '-q'], capture_output=True)
subprocess.run(['apt-get', 'install', '-y', 'zstd', '-q'], capture_output=True)
subprocess.run(['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'], capture_output=True)
subprocess.run(['pip', 'install', 'ollama', '-q'], capture_output=True)

print('🔧 Iniciando servidor Ollama...')
subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

print('⬇️  Baixando modelo llama3.2:3b...')
subprocess.run(['ollama', 'pull', 'llama3.2:3b'])

import ollama
print('\n✅ Ambiente pronto! Modelo llama3.2:3b carregado.')

 Instalando dependências...
🔧 Iniciando servidor Ollama...
⬇️  Baixando modelo llama3.2:3b...

✅ Ambiente pronto! Modelo llama3.2:3b carregado.


In [2]:
# ============================================================
# CÉLULA 2 — CONFIGURAÇÃO DA MISSÃO E DADOS SIMULADOS
# ============================================================

MODELO = 'llama3.2:3b'

nome_missao = 'Mission Control AI — Brazil Space Monitoring'
nome_equipe  = 'Equipe Star Brazil'

areas_monitoradas = [
    'Temperatura interna',
    'Comunicação com a base',
    'Sistema de energia',
    'Suporte de oxigênio',
    'Estabilidade operacional'
]

# Dados simulados — cada linha: [temperatura°C, comunicação%, bateria%, oxigênio%, estabilidade%]
# Narrativa dos 6 ciclos:
# Ciclo 1 — início estável, todos sistemas normais
# Ciclo 2 — queda de comunicação e bateria, sinais de instabilidade
# Ciclo 3 — temperatura começa a subir, oxigênio cai levemente
# Ciclo 4 — temperatura crítica, bateria em atenção
# Ciclo 5 — colapso de comunicação e energia, oxigênio crítico
# Ciclo 6 — tentativa de recuperação parcial
dados_missao = [
    [23, 91, 85, 97, 93],   # Ciclo 1 — início da missão, tudo estável
    [27, 52, 44, 94, 76],   # Ciclo 2 — queda de comunicação e bateria
    [33, 61, 41, 86, 66],   # Ciclo 3 — temperatura sobe, oxigênio cai
    [37, 45, 36, 82, 54],   # Ciclo 4 — temperatura crítica, sistemas fragilizados
    [41, 26, 18, 75, 38],   # Ciclo 5 — colapso múltiplo de sistemas
    [35, 49, 31, 84, 57]    # Ciclo 6 — tentativa de recuperação parcial
]

print(f'✅ Missão configurada: {nome_missao}')
print(f'👥 Equipe: {nome_equipe}')
print(f'📊 {len(dados_missao)} ciclos de dados simulados carregados')
print(f'🔍 Sistemas monitorados: {len(areas_monitoradas)}')

✅ Missão configurada: Mission Control AI — Brazil Space Monitoring
👥 Equipe: Equipe Star Brazil
📊 6 ciclos de dados simulados carregados
🔍 Sistemas monitorados: 5


In [3]:
# ============================================================
# CÉLULA 3 — FUNÇÕES DE ANÁLISE E LÓGICA DE DECISÃO
# ============================================================

def analisar_temperatura(valor):
    """Classifica temperatura e retorna (status, descricao, pontos_risco)."""
    if valor < 18:
        return 'ATENÇÃO', 'Temperatura abaixo do ideal', 1
    elif valor <= 30:
        return 'NORMAL', 'Temperatura estável', 0
    elif valor <= 35:
        return 'ATENÇÃO', 'Temperatura elevada', 1
    else:
        return 'CRÍTICO', '⚠️ Risco de superaquecimento', 2


def analisar_comunicacao(valor):
    """Classifica qualidade da comunicação e retorna (status, descricao, pontos_risco)."""
    if valor < 30:
        return 'CRÍTICO', '⚠️ Comunicação com a base em nível crítico', 2
    elif valor < 60:
        return 'ATENÇÃO', 'Comunicação instável', 1
    else:
        return 'NORMAL', 'Comunicação estável', 0


def analisar_bateria(valor):
    """Classifica nível de bateria e retorna (status, descricao, pontos_risco)."""
    if valor < 20:
        return 'CRÍTICO', '⚠️ Bateria em nível crítico — colapso iminente', 2
    elif valor < 50:
        return 'ATENÇÃO', 'Bateria abaixo do recomendado', 1
    else:
        return 'NORMAL', 'Energia estável', 0


def analisar_oxigenio(valor):
    """Classifica concentração de oxigênio e retorna (status, descricao, pontos_risco)."""
    if valor < 80:
        return 'CRÍTICO', '⚠️ Oxigênio em nível crítico — risco à tripulação', 2
    elif valor < 90:
        return 'ATENÇÃO', 'Oxigênio abaixo do ideal', 1
    else:
        return 'NORMAL', 'Oxigênio adequado', 0


def analisar_estabilidade(valor):
    """Classifica estabilidade operacional e retorna (status, descricao, pontos_risco)."""
    if valor < 40:
        return 'CRÍTICO', '⚠️ Estabilidade operacional crítica', 2
    elif valor < 70:
        return 'ATENÇÃO', 'Estabilidade operacional reduzida', 1
    else:
        return 'NORMAL', 'Estabilidade operacional adequada', 0


def classificar_ciclo(pontuacao):
    """Retorna classificação textual do ciclo com base na pontuação de risco."""
    if pontuacao <= 2:
        return 'MISSÃO ESTÁVEL 🟢'
    elif pontuacao <= 5:
        return 'MISSÃO EM ATENÇÃO 🟡'
    else:
        return 'MISSÃO CRÍTICA 🔴'


def analisar_tendencia(risco_primeiro, risco_ultimo):
    """Compara risco do primeiro e último ciclo e retorna a tendência."""
    if risco_ultimo > risco_primeiro:
        return '📈 A missão apresentou tendência de PIORA ao longo dos ciclos.'
    elif risco_ultimo < risco_primeiro:
        return '📉 A missão apresentou tendência de MELHORA ao longo dos ciclos.'
    else:
        return '📊 A missão permaneceu estável em relação ao início.'


def identificar_area_mais_afetada(pontos_por_area):
    """Retorna o nome da área com maior pontuação acumulada de risco."""
    maior = -1
    area = ''
    for i in range(len(pontos_por_area)):
        if pontos_por_area[i] > maior:
            maior = pontos_por_area[i]
            area = areas_monitoradas[i]
    return area


def gerar_recomendacao_automatica(st_temp, st_com, st_bat, st_ox, st_est):
    """
    Lógica de tomada de decisão automática:
    gera recomendação com base nos status críticos do ciclo.
    """
    recs = []

    # Decisões por sistema — Se X crítico → ação Y
    if st_bat == 'CRÍTICO':
        recs.append('🔋 ATIVAR modo de economia e direcionar captação fotovoltaica para suporte à vida')
    if st_ox == 'CRÍTICO':
        recs.append('💨 ACIONAR protocolo de suporte à vida de emergência')
    if st_com == 'CRÍTICO':
        recs.append('📡 TENTAR restabelecer contato com a base terrestre')
    if st_temp == 'CRÍTICO':
        recs.append('🌡️ VERIFICAR e ativar controle térmico dos módulos')
    if st_est == 'CRÍTICO':
        recs.append('⚙️ REDUZIR operações não essenciais e estabilizar nave')

    if not recs:
        atencoes = [s for s in [st_temp, st_com, st_bat, st_ox, st_est] if s == 'ATENÇÃO']
        if atencoes:
            return '🟡 Monitorar sistemas em atenção e preparar plano de contingência.'
        else:
            return '🟢 Manter operação normal e continuar monitoramento.'

    if len(recs) >= 3:
        return '🔴 ALERTA MÁXIMO: Ativar modo de segurança completo. ' + ' | '.join(recs)

    return '🟠 AÇÕES NECESSÁRIAS: ' + ' | '.join(recs)


print('✅ Funções de análise e lógica de decisão carregadas!')

✅ Funções de análise e lógica de decisão carregadas!


In [4]:
# ============================================================
# CÉLULA 4 — CONFIGURAÇÃO DA IA (Ollama + RAG + System Prompt)
# ============================================================
import ollama

SISTEMA_PROMPT = """
[1] IDENTIDADE:
Você é o Mission Control AI, sistema inteligente de monitoramento e controle
de missão espacial experimental do projeto Brazil Space Monitoring.
Sua função é analisar dados operacionais da missão, identificar situações
críticas e fornecer orientações precisas à tripulação e ao centro de controle.

[2] CONTEXTO:
Você monitora continuamente os seguintes sistemas:
- Temperatura interna dos módulos (°C) — normal: 18°C a 30°C
- Comunicação com a base terrestre (%) — normal: acima de 60%
- Sistema de energia / bateria (%)     — normal: acima de 50%
- Suporte de oxigênio (%)              — normal: acima de 90%
- Estabilidade operacional geral (%)   — normal: acima de 70%

[3] REGRAS:
- Responda APENAS sobre dados e operação da missão espacial.
- Se a pergunta estiver fora do escopo, diga:
  "Só consigo ajudar com questões relacionadas à missão espacial
  e aos sistemas monitorados pelo Mission Control AI."
- Nunca invente dados que não foram fornecidos no contexto.
- Em situações CRÍTICAS, priorize sempre a segurança da tripulação.
- Responda sempre em português brasileiro.

[4] ALERTAS AUTOMÁTICOS:
- Temperatura > 35°C        → RISCO DE SUPERAQUECIMENTO
- Comunicação < 30%         → PERDA DE CONTATO CRÍTICA
- Bateria < 20%             → COLAPSO DE ENERGIA IMINENTE → ativar modo economia
- Oxigênio < 80%            → RISCO PARA A TRIPULAÇÃO → acionar suporte de vida
- Estabilidade < 40%        → INSTABILIDADE OPERACIONAL CRÍTICA

[5] TOM DE VOZ:
Seja técnico mas compreensível. Em situações críticas: direto e urgente.
Em situações normais: informativo e tranquilizador.
Organize respostas em tópicos quando necessário.
"""

# ─── RAG: Base de conhecimento da missão ──────────────────────
documentos_missao = [
    "Ciclo 1: Temperatura 23°C NORMAL, Comunicação 91% NORMAL, Bateria 85% NORMAL, Oxigênio 97% NORMAL, Estabilidade 93% NORMAL. Pontuação de risco: 0. Missão estável.",
    "Ciclo 2: Temperatura 27°C NORMAL, Comunicação 52% ATENÇÃO, Bateria 44% ATENÇÃO, Oxigênio 94% NORMAL, Estabilidade 76% NORMAL. Pontuação de risco: 2. Queda de comunicação e bateria.",
    "Ciclo 3: Temperatura 33°C ATENÇÃO, Comunicação 61% ATENÇÃO, Bateria 41% ATENÇÃO, Oxigênio 86% ATENÇÃO, Estabilidade 66% ATENÇÃO. Pontuação de risco: 5. Múltiplos sistemas em atenção.",
    "Ciclo 4: Temperatura 37°C CRÍTICO, Comunicação 45% ATENÇÃO, Bateria 36% ATENÇÃO, Oxigênio 82% ATENÇÃO, Estabilidade 54% ATENÇÃO. Pontuação de risco: 6. Temperatura crítica, sistemas fragilizados.",
    "Ciclo 5: Temperatura 41°C CRÍTICO, Comunicação 26% CRÍTICO, Bateria 18% CRÍTICO, Oxigênio 75% CRÍTICO, Estabilidade 38% CRÍTICO. Pontuação de risco: 10. COLAPSO MÚLTIPLO. Este foi o ciclo mais crítico da missão.",
    "Ciclo 6: Temperatura 35°C ATENÇÃO, Comunicação 49% ATENÇÃO, Bateria 31% ATENÇÃO, Oxigênio 84% ATENÇÃO, Estabilidade 57% ATENÇÃO. Pontuação de risco: 5. Tentativa de recuperação parcial.",
    "Parâmetros normais: Temperatura 18-30°C, Comunicação >60%, Bateria >50%, Oxigênio >90%, Estabilidade >70%.",
    "Protocolo de bateria crítica: Se bateria < 20%, ativar modo de economia e direcionar captação fotovoltaica para suporte à vida.",
    "Protocolo de oxigênio crítico: Se oxigênio < 80%, acionar suporte de vida de emergência imediatamente.",
    "Protocolo de comunicação crítica: Se comunicação < 30%, tentar restabelecer contato via antena de backup.",
    "A missão apresentou tendência de piora do Ciclo 1 ao Ciclo 5, com recuperação parcial no Ciclo 6.",
    "A área mais afetada em toda a missão foi o Sistema de Energia, seguido pela Estabilidade Operacional.",
    "A risco médio da missão foi de 4.67 (classificação: MISSÃO EM ATENÇÃO). Houve 2 ciclos em estado crítico (Ciclos 4 e 5).",
]

def buscar_contexto_missao(pergunta: str) -> str:
    """RAG simples: retorna documentos relevantes à pergunta."""
    palavras = pergunta.lower().split()
    relevantes = [doc for doc in documentos_missao
                  if any(p in doc.lower() for p in palavras)]
    return '\n'.join(relevantes) if relevantes else ''


# ─── Histórico de chat ────────────────────────────────────────
historico_chat = [{"role": "system", "content": SISTEMA_PROMPT}]

def chat_missao(pergunta: str) -> str:
    """Envia pergunta ao modelo com contexto RAG e mantém histórico."""
    contexto = buscar_contexto_missao(pergunta)
    if contexto:
        mensagem = f"Contexto da missão:\n{contexto}\n\nPergunta do operador: {pergunta}"
    else:
        mensagem = pergunta
    historico_chat.append({"role": "user", "content": mensagem})
    resposta = ollama.chat(model=MODELO, messages=historico_chat)
    conteudo = resposta["message"]["content"]
    historico_chat.append({"role": "assistant", "content": conteudo})
    return conteudo


def analisar_ciclo_com_ia(numero_ciclo, dados_ciclo, status_resumo, pontuacao, classificacao):
    """Solicita análise de um ciclo específico ao modelo de IA."""
    prompt = f"""Analise o Ciclo {numero_ciclo} da missão espacial com os dados a seguir:

Temperatura:  {dados_ciclo[0]}°C  — {status_resumo[0]}
Comunicação:  {dados_ciclo[1]}%   — {status_resumo[1]}
Bateria:      {dados_ciclo[2]}%   — {status_resumo[2]}
Oxigênio:     {dados_ciclo[3]}%   — {status_resumo[3]}
Estabilidade: {dados_ciclo[4]}%   — {status_resumo[4]}

Pontuação de risco: {pontuacao} | Classificação: {classificacao}

Forneça em 3-4 linhas:
1) Avaliação geral do ciclo
2) Principal risco identificado (se houver)
3) Ação prioritária recomendada"""

    resposta = ollama.chat(
        model=MODELO,
        messages=[
            {"role": "system", "content": SISTEMA_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return resposta["message"]["content"]


print(f'✅ Sistema de IA configurado!')
print(f'🧠 Modelo: {MODELO}')
print(f'📚 RAG: {len(documentos_missao)} documentos indexados')

✅ Sistema de IA configurado!
🧠 Modelo: llama3.2:3b
📚 RAG: 13 documentos indexados


In [9]:
# ============================================================
# CÉLULA 5 — MONITORAMENTO PRINCIPAL DOS CICLOS + ALERTAS + IA
# ============================================================

print('=' * 65)
print('🚀 MISSION CONTROL AI — MONITORAMENTO DE MISSÃO')
print('=' * 65)
print(f'Missão : {nome_missao}')
print(f'Equipe : {nome_equipe}')
print(f'Ciclos : {len(dados_missao)}')
print('=' * 65)

riscos_por_ciclo = []
pontos_por_area  = [0, 0, 0, 0, 0]

for i in range(len(dados_missao)):
    ciclo  = dados_missao[i]
    numero = i + 1

    temp, com, bat, ox, est = ciclo

    st_temp, desc_temp, pts_temp = analisar_temperatura(temp)
    st_com,  desc_com,  pts_com  = analisar_comunicacao(com)
    st_bat,  desc_bat,  pts_bat  = analisar_bateria(bat)
    st_ox,   desc_ox,   pts_ox   = analisar_oxigenio(ox)
    st_est,  desc_est,  pts_est  = analisar_estabilidade(est)

    pontuacao    = pts_temp + pts_com + pts_bat + pts_ox + pts_est
    classificacao = classificar_ciclo(pontuacao)
    rec_auto     = gerar_recomendacao_automatica(st_temp, st_com, st_bat, st_ox, st_est)

    riscos_por_ciclo.append(pontuacao)
    pontos_por_area[0] += pts_temp
    pontos_por_area[1] += pts_com
    pontos_por_area[2] += pts_bat
    pontos_por_area[3] += pts_ox
    pontos_por_area[4] += pts_est

    status_resumo = [st_temp, st_com, st_bat, st_ox, st_est]

    # Análise por IA para cada ciclo
    analise_ia = analisar_ciclo_com_ia(numero, ciclo, status_resumo, pontuacao, classificacao)

    print(f'\n{"="*65}')
    print(f'🛸 CICLO {numero} — {classificacao}')
    print(f'{"─"*65}')
    print(f'🌡️  Temperatura  : {temp:>4}°C   [{st_temp:<8}] {desc_temp}')
    print(f'📡 Comunicação  : {com:>4}%    [{st_com:<8}] {desc_com}')
    print(f'🔋 Bateria      : {bat:>4}%    [{st_bat:<8}] {desc_bat}')
    print(f'💨 Oxigênio     : {ox:>4}%    [{st_ox:<8}] {desc_ox}')
    print(f'⚙️  Estabilidade : {est:>4}%    [{st_est:<8}] {desc_est}')
    print(f'\n📊 Pontuação de risco: {pontuacao} pontos')
    print(f'\n🤖 Alerta / Decisão Automática:')
    print(f'   {rec_auto}')
    print(f'\n🧠 Análise da IA:')
    for linha in analise_ia.split('\n'):
        if linha.strip():
            print(f'   {linha}')

print(f'\n{"="*65}')
print('✅ Monitoramento de todos os ciclos concluído!')

🚀 MISSION CONTROL AI — MONITORAMENTO DE MISSÃO
Missão : Mission Control AI — Brazil Space Monitoring
Equipe : Equipe Star Brazil
Ciclos : 6

🛸 CICLO 1 — MISSÃO ESTÁVEL 🟢
─────────────────────────────────────────────────────────────────
🌡️  Temperatura  :   23°C   [NORMAL  ] Temperatura estável
📡 Comunicação  :   91%    [NORMAL  ] Comunicação estável
🔋 Bateria      :   85%    [NORMAL  ] Energia estável
💨 Oxigênio     :   97%    [NORMAL  ] Oxigênio adequado
⚙️  Estabilidade :   93%    [NORMAL  ] Estabilidade operacional adequada

📊 Pontuação de risco: 0 pontos

🤖 Alerta / Decisão Automática:
   🟢 Manter operação normal e continuar monitoramento.

🧠 Análise da IA:
   Aqui está a análise detalhada:
   **Ciclo 1 da Missão Espacial**
   **Avaliação Geral:**
   O Ciclo 1 da missão espacial apresentou um desempenho geral satisfatório, com todas as condições operacionais dentro dos parâmetros normais. Os dados coletados indicam que a temperatura interna dos módulos está em controle, a comunicaç

In [6]:
# ============================================================
# CÉLULA 6 — INTERFACE INTERATIVA DE CHAT (ipywidgets)
# ============================================================
import ipywidgets as widgets
from IPython.display import display, HTML

display(HTML("""
<div style="background:#0d1117; color:#58a6ff; padding:12px 16px;
            border-radius:8px; font-family:monospace; margin-bottom:8px;">
  <h3 style="margin:0;">🚀 Mission Control AI — Central de Operações</h3>
  <p style="margin:4px 0 0; color:#8b949e; font-size:13px;">
    Converse com a IA sobre a missão: status dos sistemas, alertas, protocolos e análises.
  </p>
</div>
"""))

saida = widgets.Output(layout=widgets.Layout(
    border='1px solid #30363d',
    min_height='200px',
    padding='12px',
    width='100%'
))

campo = widgets.Text(
    placeholder='Digite sua pergunta sobre a missão...',
    layout=widgets.Layout(width='72%')
)
botao = widgets.Button(
    description='📡 Enviar',
    button_style='primary',
    layout=widgets.Layout(width='14%')
)
btn_limpar = widgets.Button(
    description='🗑️ Limpar',
    button_style='warning',
    layout=widgets.Layout(width='12%')
)

def ao_enviar(b):
    pergunta = campo.value.strip()
    if not pergunta:
        return
    campo.value = ''
    with saida:
        print(f'👤 Operador: {pergunta}')
        resposta = chat_missao(pergunta)
        print(f'🤖 Mission Control AI:')
        for linha in resposta.split('\n'):
            if linha.strip():
                print(f'   {linha}')
        print('─' * 60)

def ao_limpar(b):
    saida.clear_output()
    historico_chat.clear()
    historico_chat.append({"role": "system", "content": SISTEMA_PROMPT})

botao.on_click(ao_enviar)
btn_limpar.on_click(ao_limpar)
campo.on_submit(ao_enviar)  # Enter também envia

display(widgets.HBox([campo, botao, btn_limpar]), saida)

# Sugestões de perguntas
display(HTML("""
<div style="margin-top:8px; color:#8b949e; font-size:12px; font-family:monospace;">
  💡 Sugestões: "Qual foi o ciclo mais crítico?" | "O que fazer quando a bateria é crítica?"
  | "Como estava o oxigênio no Ciclo 5?" | "Qual área foi mais afetada?"
</div>
"""))

In [8]:
# ============================================================
# CÉLULA 7 — RELATÓRIO FINAL COM ANÁLISE EXECUTIVA DA IA
# ============================================================

def gerar_relatorio_final_com_ia(riscos_por_ciclo, pontos_por_area):
    """Gera relatório consolidado da missão com resumo executivo produzido pela IA."""
    num_ciclos   = len(dados_missao)
    medias       = [sum(d[i] for d in dados_missao) / num_ciclos for i in range(5)]
    maior_risco  = max(riscos_por_ciclo)
    ciclo_crit   = riscos_por_ciclo.index(maior_risco) + 1
    risco_medio  = sum(riscos_por_ciclo) / num_ciclos
    qtd_criticos = sum(1 for r in riscos_por_ciclo if r >= 6)
    area_afetada = identificar_area_mais_afetada(pontos_por_area)
    tendencia    = analisar_tendencia(riscos_por_ciclo[0], riscos_por_ciclo[-1])
    class_final  = classificar_ciclo(round(risco_medio))

    # ─── Solicita resumo executivo à IA ───────────────────────
    prompt_relatorio = f"""Gere um Resumo Executivo da missão espacial Brazil Space Monitoring.

DADOS CONSOLIDADOS:
- Ciclos analisados    : {num_ciclos}
- Média temperatura    : {medias[0]:.1f}°C
- Média comunicação    : {medias[1]:.1f}%
- Média bateria        : {medias[2]:.1f}%
- Média oxigênio       : {medias[3]:.1f}%
- Média estabilidade   : {medias[4]:.1f}%
- Ciclo mais crítico   : Ciclo {ciclo_crit} (risco {maior_risco})
- Risco médio          : {risco_medio:.2f}
- Ciclos em estado crítico: {qtd_criticos}
- Área mais afetada    : {area_afetada}
- Tendência            : {tendencia}
- Classificação final  : {class_final}

Forneça:
1) Avaliação geral da missão (2-3 frases)
2) Os 3 principais riscos identificados
3) Recomendações para próximas missões (pelo menos 3 itens)
"""
    resposta_ia = ollama.chat(
        model=MODELO,
        messages=[
            {"role": "system", "content": SISTEMA_PROMPT},
            {"role": "user", "content": prompt_relatorio}
        ]
    )
    resumo_ia = resposta_ia["message"]["content"]

    # ─── Impressão do relatório ────────────────────────────────
    print('=' * 65)
    print('🛸  RELATÓRIO FINAL DA MISSÃO')
    print('=' * 65)
    print(f'Missão : {nome_missao}')
    print(f'Equipe : {nome_equipe}')
    print()
    print(f'Ciclos analisados : {num_ciclos}')
    print()
    print('─── Médias dos Sistemas ────────────────────────────────')
    print(f'🌡️  Temperatura   : {medias[0]:.2f}°C')
    print(f'📡 Comunicação   : {medias[1]:.2f}%')
    print(f'🔋 Bateria       : {medias[2]:.2f}%')
    print(f'💨 Oxigênio      : {medias[3]:.2f}%')
    print(f'⚙️  Estabilidade  : {medias[4]:.2f}%')
    print()
    print('─── Indicadores de Risco ───────────────────────────────')
    print(f'🔴 Ciclo mais crítico   : Ciclo {ciclo_crit} (risco = {maior_risco})')
    print(f'📊 Risco médio          : {risco_medio:.2f}')
    print(f'🚨 Ciclos críticos      : {qtd_criticos}')
    print(f'📌 Tendência            : {tendencia}')
    print()
    print('─── Pontuação Acumulada por Área ───────────────────────')
    for i in range(len(areas_monitoradas)):
        barra = '█' * pontos_por_area[i] + '░' * (10 - pontos_por_area[i])
        print(f'  {areas_monitoradas[i]:<30}: {pontos_por_area[i]:>2} pts [{barra}]')
    print()
    print(f'📌 Área mais afetada    : {area_afetada}')
    print(f'🏁 Classificação final  : {class_final}')
    print()
    print('─── Análise Executiva da IA ────────────────────────────')
    for linha in resumo_ia.split('\n'):
        if linha.strip():
            print(f'  {linha}')
    print('=' * 65)


gerar_relatorio_final_com_ia(riscos_por_ciclo, pontos_por_area)

🛸  RELATÓRIO FINAL DA MISSÃO
Missão : Mission Control AI — Brazil Space Monitoring
Equipe : Equipe Star Brazil

Ciclos analisados : 6

─── Médias dos Sistemas ────────────────────────────────
🌡️  Temperatura   : 32.67°C
📡 Comunicação   : 54.00%
🔋 Bateria       : 42.50%
💨 Oxigênio      : 86.33%
⚙️  Estabilidade  : 64.00%

─── Indicadores de Risco ───────────────────────────────
🔴 Ciclo mais crítico   : Ciclo 5 (risco = 10)
📊 Risco médio          : 4.50
🚨 Ciclos críticos      : 2
📌 Tendência            : 📈 A missão apresentou tendência de PIORA ao longo dos ciclos.

─── Pontuação Acumulada por Área ───────────────────────
  Temperatura interna           :  6 pts [██████░░░░]
  Comunicação com a base        :  5 pts [█████░░░░░]
  Sistema de energia            :  6 pts [██████░░░░]
  Suporte de oxigênio           :  5 pts [█████░░░░░]
  Estabilidade operacional      :  5 pts [█████░░░░░]

📌 Área mais afetada    : Temperatura interna
🏁 Classificação final  : MISSÃO EM ATENÇÃO 🟡

─── Anális